# 1. Data Preparation

**Pipeline position:** first prep notebook. Reads the cleaned panel from the
main capstone repo (`Master.csv`, `clusters1995.csv`) and produces the panel
that every chart downstream consumes.

## What this notebook does

1. Loads `Master.csv` and applies the forest-adjusted sample selection
   (subtract forestry rents from total NR rents, keep countries with
   adjusted rents at or above 1% of GDP, force-include Gulf states).
2. Restricts to 1995–2019 and merges k=5 cluster labels.
3. Adds the derived columns the charts need (log GDP per capita, production
   per capita, scaled bubble size for the animated scatter, lagged ECI,
   log transforms, mean-centred interaction terms).
4. Saves the resulting panel as `panel.parquet` and the 1995 cluster
   assignments as `clusters_1995.csv`.

## Outputs

- `artifacts/panel.parquet` — country–year panel for the 38-country sample,
  with all derived columns. Used by every other notebook and every chart.
- `artifacts/clusters_1995.csv` — `Country Code, Country, ClusterLabels`
  for the cluster world map (chart 5).
- `artifacts/production_panel_world.csv` — every country with production data, 1995–2019, in long format (Resource × Year × Country) with USD value, USD per capita, and % of GDP. Used by the Resource Production Intensity world map (chart 1).
- `artifacts/sample.csv` — the list of 38 country codes, written so the
  robustness notebook can rebuild larger samples without re-doing this work.

Run from `capstone_viz/prep/`. Expects `_config.py` in the same folder.

In [1]:
# ── Imports and config ──────────────────────────────────────────────────────
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath('')) + '/prep')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd

from _config import (
    INTERMEDIARY, ARTIFACTS, GULF_STATES, NR_THRESHOLD, HIGH_INCOME_1995,
    YEAR_MIN, YEAR_MAX, LOG_COLS, ECI_COL,
    artifact_path, load_master, load_clusters_1995,
)

print(f'Reading from: {INTERMEDIARY}')
print(f'Writing to:   {ARTIFACTS}')


Reading from: /Users/leoss/Desktop/GitHub/leoss14.github.io/projects/capstone/New code/intermediary
Writing to:   /Users/leoss/Desktop/GitHub/leoss14.github.io/projects/capstone/New code/threshold_sweep/artifacts_t0


## Sample selection

The sample is defined on the 1995 snapshot. For each country we compute
adjusted natural-resource rents (total NR rents minus forestry rents,
clipped at zero) and keep those at or above 1% of GDP. We then drop
countries classified as high-income by WDI in 1995, which removes
Norway, Australia, Canada and similar advanced economies that pass the
rent threshold but lie outside the resource-rich developing-country focus
of the analysis. Gulf states are then force-included: their 1995
adjusted rents understate the oil sectors that come online later in the
panel, and several were already high-income in 1995, so the income filter
would otherwise drop them.


In [2]:
master = load_master()

base_1995 = master[master['Year'] == 1995].copy()
base_1995['NR_adj'] = (
    base_1995['Total natural resources rents (% of GDP)'].fillna(0)
    - base_1995['Forestry rents (% of GDP)'].fillna(0)
).clip(lower=0)

# Step 1: countries with adjusted NR rents at or above the threshold
include_lst = base_1995.loc[
    base_1995['NR_adj'] >= NR_THRESHOLD, 'Country Code'
].unique().tolist()
n_after_threshold = len(include_lst)

# Step 2: drop 1995 high-income economies (developing-country focus)
dropped_hi = [c for c in include_lst if c in HIGH_INCOME_1995]
include_lst = [c for c in include_lst if c not in HIGH_INCOME_1995]

# Step 3: force-include Gulf states (overrides the income filter for the six
# Gulf economies; their oil-sector trajectories are part of the resource-curse
# question even where income status excludes them)
for g in GULF_STATES:
    if g not in include_lst:
        include_lst.append(g)

print(f'After 1% threshold:       {n_after_threshold} countries')
print(f'After income filter:      {n_after_threshold - len(dropped_hi)} countries '
      f'(dropped {len(dropped_hi)}: {sorted(dropped_hi)})')
print(f'After Gulf force-include: {len(include_lst)} countries')


After 1% threshold:       131 countries
After income filter:      103 countries (dropped 28: ['ARE', 'AUS', 'AUT', 'BHR', 'BRB', 'BRN', 'CAN', 'CHE', 'CYP', 'DEU', 'DNK', 'ESP', 'FIN', 'FRA', 'GBR', 'GRC', 'IRL', 'ISR', 'ITA', 'JPN', 'KWT', 'NLD', 'NOR', 'NZL', 'PRT', 'QAT', 'SWE', 'USA'])
After Gulf force-include: 107 countries


## Build the panel

Restrict `Master.csv` to the sample, attach cluster labels, add the columns
charts will need, and save.


In [3]:
clusters_1995 = load_clusters_1995()
cl_map = clusters_1995[['Country Code', 'ClusterLabels']].drop_duplicates('Country Code')

panel = master[
    master['Year'].between(YEAR_MIN, YEAR_MAX)
    & master['Country Code'].isin(include_lst)
].copy()

panel = panel.sort_values(['Country Code', 'Year']).reset_index(drop=True)
panel = panel.merge(cl_map, on='Country Code', how='left')

# Derived columns used by the descriptive and animation charts ----------------
panel['Log_GDP_pc'] = np.log(
    panel['GDP per capita (constant prices, PPP)'].replace(0, np.nan)
)
panel['Total_Production_Value_Per_Capita'] = (
    panel['Total_Production_Value'] / panel['Population'].replace(0, np.nan)
)
panel['Prod_pc'] = panel['Total_Production_Value_Per_Capita']
panel['Bubble']  = np.sqrt(panel['Prod_pc'].fillna(0))

bmin, bmax = panel['Bubble'].min(), panel['Bubble'].max()
panel['Bubble_Scaled'] = 8 + (panel['Bubble'] - bmin) / (bmax - bmin + 1e-9) * 42

# Lagged ECI ------------------------------------------------------------------
panel['L1_ECI'] = panel.groupby('Country Code')[ECI_COL].shift(1)

# Log-transform features used by the ML and regression notebooks --------------
for c in LOG_COLS:
    if c in panel.columns:
        panel[c] = np.log1p(panel[c]).replace([np.inf, -np.inf], np.nan)

# Aliases used by the regression notebook (already log-transformed above) -----
panel['log_HCI']             = panel['Human capital index']
panel['log_GFCF']            = panel['Gross fixed capital formation, all, Constant prices, Percent of GDP']
panel['log_Production_Value']= panel['Total_Production_Value_Per_Capita']

# Mean-centred interaction terms (regression notebook + ML notebook) ----------
hci_m  = panel['log_HCI'].mean()
gfcf_m = panel['log_GFCF'].mean()
prod_m = panel['log_Production_Value'].mean()
frt_m  = panel['Forestry rents (% of GDP)'].mean()

panel['HCI_x_ProductionValue']  = (panel['log_HCI']  - hci_m)  * (panel['log_Production_Value'] - prod_m)
panel['GFCF_x_ProductionValue'] = (panel['log_GFCF'] - gfcf_m) * (panel['log_Production_Value'] - prod_m)
panel['log_HCI_x_log_Production']  = panel['HCI_x_ProductionValue']
panel['log_GFCF_x_log_Production'] = panel['GFCF_x_ProductionValue']
panel['log_HCI_x_forestry_rents']  = (panel['log_HCI']  - hci_m)  * (panel['Forestry rents (% of GDP)'] - frt_m)
panel['log_GFCF_x_forestry_rents'] = (panel['log_GFCF'] - gfcf_m) * (panel['Forestry rents (% of GDP)'] - frt_m)

print(f'Panel: {panel["Country Code"].nunique()} countries, '
      f'{len(panel):,} obs, {YEAR_MIN}-{YEAR_MAX}')


Panel: 107 countries, 2,675 obs, 1995-2019


## Production panel for the world map

The Resource Production Intensity choropleth (chart 1) needs the
**full set of countries with production data**, not just the 49
forest-adjusted ones in the panel above. It also needs values per
resource (Oil, Natural Gas, Coal, Metals) per year, not a single
annual mean. We build that artifact here, separately from the
panel, by reading `NaturalResource.csv` (long format with
`Production_TotalValue` already in USD) plus the population and
GDP columns from `Master.csv` for the per-capita and %-of-GDP
normalisations.

Resource buckets:

- `Total` — sum across all resources for each country–year
- `Oil`, `Natural Gas`, `Coal` — kept as-is
- `Metals` — every other resource (industrial, precious, battery,
  rare-earth) summed into one bucket, matching the convention used
  in the portfolio.

In [4]:
# ── Production panel for the world map (chart 1) ────────────────────────────
# Reads NaturalResource.csv via load_natural_resource() in _config. The file
# is in wide format with separate Production / Reserves columns (no Metric
# column to filter on), so we use Production_TotalValue directly. Falls back
# to a Total-only build from Master.csv if the long file isn't available.
try:
    from _config import load_natural_resource
    nr = load_natural_resource()
    have_long = True
except (FileNotFoundError, ImportError):
    have_long = False

if have_long:
    # Bucket: keep Oil / Natural Gas / Coal as-is, lump everything else into
    # Metals. Matching by Resource name handles the Coal-as-Hydrocarbon
    # category quirk; any unrecognised resource ends up in Metals.
    def _bucket(r):
        if r in ("Oil", "Natural Gas", "Coal"):
            return r
        return "Metals"
    nr = nr.copy()
    nr["Bucket"] = nr["Resource"].apply(_bucket)

    # Pick the value column that actually exists. The wide file uses
    # Production_TotalValue; older long-format pipelines used Value.
    val_col = ("Production_TotalValue"
               if "Production_TotalValue" in nr.columns
               else "Value")
    nr = nr.dropna(subset=[val_col, "Country Code", "Year"])
    nr["Year"] = nr["Year"].astype(int)

    by_bucket = (nr.groupby(["Country Code", "Year", "Bucket"], as_index=False)[val_col]
                   .sum()
                   .rename(columns={val_col: "prod_usd"}))

    total = (nr.groupby(["Country Code", "Year"], as_index=False)[val_col]
               .sum()
               .rename(columns={val_col: "prod_usd"}))
    total["Bucket"] = "Total"
    long_df = pd.concat([by_bucket, total[by_bucket.columns]], ignore_index=True)
else:
    print("  (NaturalResource.csv not found — building Total-only fallback)")
    long_df = (master.dropna(subset=["Total_Production_Value"])
                     [["Country Code", "Year", "Total_Production_Value"]]
                     .rename(columns={"Total_Production_Value": "prod_usd"}))
    long_df["Bucket"] = "Total"

# Attach country name, population, GDP from Master.csv (for normalisations)
gdp_col = ("GDP, current prices (PPP)"
           if "GDP, current prices (PPP)" in master.columns
           else "GDP per capita (constant prices, PPP)")
meta_cols = ["Country Code", "Country Name", "Year", "Population", gdp_col]
meta_cols = [c for c in meta_cols if c in master.columns]
meta = master[meta_cols].drop_duplicates(["Country Code", "Year"])

long_df = long_df.merge(meta, on=["Country Code", "Year"], how="left")
long_df = long_df[long_df["Year"].between(YEAR_MIN, YEAR_MAX)]

# Per-capita and %-of-GDP
long_df["prod_pc"] = (long_df["prod_usd"]
                      / long_df["Population"].replace(0, np.nan))
if gdp_col == "GDP, current prices (PPP)":
    long_df["prod_pct_gdp"] = 100.0 * long_df["prod_usd"] / long_df[gdp_col].replace(0, np.nan)
else:
    gdp_total = long_df[gdp_col] * long_df["Population"]
    long_df["prod_pct_gdp"] = 100.0 * long_df["prod_usd"] / gdp_total.replace(0, np.nan)

long_df = (long_df.dropna(subset=["Country Code"])
                  .reset_index(drop=True))

BUCKET_ORDER = ["Total", "Oil", "Natural Gas", "Coal", "Metals"]
long_df["Bucket"] = pd.Categorical(long_df["Bucket"],
                                   categories=BUCKET_ORDER, ordered=True)
long_df = long_df.sort_values(["Bucket", "Year", "Country Code"]).reset_index(drop=True)

long_df.to_csv(artifact_path('production_panel_world.csv'), index=False)
n_c = long_df['Country Code'].nunique()
n_y = long_df['Year'].nunique()
n_b = long_df['Bucket'].nunique()
print(f'  Saved production_panel_world.csv '
      f'({n_c} countries, {n_y} years, {n_b} buckets, {len(long_df):,} rows)')

  Saved production_panel_world.csv (155 countries, 25 years, 5 buckets, 12,124 rows)


## Save artifacts

Three files. The parquet is the heavy one (panel) and is read by every
other prep notebook. The two CSVs are tiny and used directly by the viz
notebooks.


In [5]:
panel.to_parquet(artifact_path('panel.parquet'), index=False)
print(f'  Saved panel.parquet ({len(panel):,} rows, {panel.shape[1]} cols)')

clusters_1995[['Country Code', 'Country', 'ClusterLabels']] \
    .drop_duplicates('Country Code') \
    .to_csv(artifact_path('clusters_1995.csv'), index=False)
print(f'  Saved clusters_1995.csv')

pd.Series(include_lst, name='Country Code') \
    .to_csv(artifact_path('sample.csv'), index=False)
print(f'  Saved sample.csv ({len(include_lst)} countries)')


  Saved panel.parquet (2,675 rows, 65 cols)
  Saved clusters_1995.csv
  Saved sample.csv (107 countries)


## Summary

| Step | Result |
|---|---|
| Master.csv loaded | full panel, all countries, 1995–2019 |
| Forest-adjusted threshold (≥1% + Gulf) | 38 countries kept |
| Derived columns added | log GDP per capita, production per capita, lagged ECI, log transforms, 4 interaction terms |
| Saved | panel.parquet, clusters_1995.csv, sample.csv, production_panel_world.csv |

Run notebook 2 (`2_clusters.ipynb`) next.